# SQL Data Extraction & KPI Engineering

Welcome to the SQL analysis section of my project. In this document, I connect to my cloud-based PostgreSQL database (hosted on Neon) to perform advanced data extraction, cleaning, and aggregation. 

The goal of these queries is to transform raw clinical and lifestyle data into actionable insights, preparing the datasets that will power my interactive Tableau dashboards. I have divided my SQL analysis into two main pillars: **Clinical Demographics** to understand my patient cohort, and **Key Performance Indicators (KPIs)** to evaluate survival and risk factors.

---

## Part 1: Clinical Demographics (TCGA Cohort)
In this section, I prepare the demographic foundations of the study by extracting key features from the patients dataset.

* **Demo 1: Cancer Type & Sex Distribution:** I select cancer_type and sex to establish the baseline population for the analysis.
* **Demo 2: Age Binning Logic:** I use CASE WHEN to transform the continuous diagnosis_age variable into four categorical buckets ('1. Under 40', '2. 40-59', '3. 60-79', '4. 80+'). This categorization is essential to visualize how cancer incidence shifts across different age groups.
* **Demo 3: Data Filtering:** I filter the cohort to exclude null values in the sex column, ensuring the integrity and reliability of the demographic analysis.

---

## Part 2: Lifestyle & Population KPIs (NHANES Cohort)
This section performs feature engineering on raw survey data to create clean, analytical KPIs focused on lifestyle and survival.

* **KPI 1: Activity Level Categorization:** I map the raw vigorous_recreation codes into a human-readable format, classifying patients as 'Active' (1) or 'Sedentary' (2) to simplify the analysis of survival trends.
* **KPI 2: Sedentary Threshold Engineering:** I categorize the continuous sedentary_minutes_day variable into three distinct tiers (<4 hours, 4-8 hours, >8 hours). This allows for a direct comparison of how different levels of daily inactivity correlate with survival_months.
* **KPI 3: Smoking Habit Normalization:** I standardize the current_smoking_status field by grouping codes (1, 2) into a unified 'Smoker' category and (3) into 'Non-Smoker'. This ensures that the dataset is ready for high-level survival analysis and mortality status evaluation.



# Demographics

In [ ]:
SELECT 
    cancer_type,
    sex,
    CASE 
        WHEN CAST(diagnosis_age AS NUMERIC) < 40 THEN '1. Under 40'
        WHEN CAST(diagnosis_age AS NUMERIC) BETWEEN 40 AND 59 THEN '2. 40-59'
        WHEN CAST(diagnosis_age AS NUMERIC) BETWEEN 60 AND 79 THEN '3. 60-79'
        ELSE '4. 80+'
    END AS age_group
FROM patients
WHERE sex IS NOT NULL;


# KPIs:

In [ ]:
SELECT 
    age,
    gender,
    CASE 
        WHEN vigorous_recreation = 1 THEN 'Active'
        WHEN vigorous_recreation = 2 THEN 'Sedentary'
        ELSE 'Unknown'
    END AS activity_level,
    CASE 
        WHEN CAST(sedentary_minutes_day AS NUMERIC) < 240 THEN '<4 hours'
        WHEN CAST(sedentary_minutes_day AS NUMERIC) BETWEEN 240 AND 480 THEN '4-8 hours'
        WHEN CAST(sedentary_minutes_day AS NUMERIC) > 480 THEN '>8 hours'
        ELSE 'Unknown'
    END AS sedentary_category,
    CASE 
        WHEN current_smoking_status IN (1, 2) THEN 'Smoker'
        WHEN current_smoking_status = 3 THEN 'Non-Smoker'
        ELSE 'Unknown'
    END AS smoking_status,
    survival_months,
    mortality_status
FROM nhanes_analytics_data
WHERE survival_months IS NOT NULL;